# Семинар. Решение нелинейных уравнений

**Курс «Численные методы», 2 курс физического факультета СПбГУ**

**План семинара**

Разминка: интерполяция

0. Немного Python
1. Постановка задачи и локализация корней
2. Метод деления пополам
3. Метод простой итерации
4. Метод Ньютона
5. Метод секущих
6. Сравнение методов. Метод Брента и `scipy`
7. Задачи
8. Домашнее задание

Дополнительно: метод парабол, поиск всех корней.

**Напоминание по работе с блокнотом**

- Ячейка выполняется сочетанием **Shift + Enter**. Ячейки выполняются в том порядке, в котором вы их запускаете.
- Ошибка `NameError: name '...' is not defined` обычно означает, что пропущена одна из ячеек выше.
- Если ячейка выполняется слишком долго (например, бесконечный цикл), остановите её: кнопка «стоп» слева от ячейки или меню **Runtime → Interrupt execution** в Colab (**Kernel → Interrupt** в Jupyter).
- Ячейки **«Пишем вместе»** заполняем вместе на семинаре. Ячейки с готовым кодом нужно просто запустить.

Проверим, что нужные библиотеки установлены. Если какая-то не установлена, выполните в отдельной ячейке команду `%pip install имя_библиотеки` и перезапустите блокнот (в Colab всё уже установлено).

In [ ]:
import importlib.util

for name in ['numpy', 'scipy', 'matplotlib']:
    if importlib.util.find_spec(name) is None:
        print(f'{name}: НЕ установлен, выполните  %pip install {name}')
    else:
        print(f'{name}: OK')

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams['figure.figsize'] = (8, 4.5)
plt.rcParams['axes.grid'] = True

## Разминка: интерполяция

Дана таблица значений функции $\sin x$ в 6 равноотстоящих точках на $[0, \pi]$ (просто запустите ячейку).

In [ ]:
from scipy.interpolate import CubicSpline

x_table = np.linspace(0, np.pi, 6)
y_table = np.sin(x_table)
print(x_table)
print(y_table)

1. Вычислите значение в точке $x = 1$ с помощью кусочно-линейной интерполяции (`np.interp`) и сравните с `np.sin(1.0)`.
2. Постройте по таблице кубический сплайн (назовите его `cs_table`, он понадобится в конце семинара) и вычислите его значение в точке $x = 1$. Какая интерполяция точнее?
3. Нарисуйте табличные точки, оба интерполянта и горизонтальную прямую $y = 0.5$. При каком $x$ сплайн равен $0.5$?

In [ ]:
# Ваш код


## 0. Немного Python

Сегодня понадобятся три вещи: передача функции в другую функцию, списки и цикл с досрочным выходом.

**Функция как аргумент.** Функцию можно передать в другую функцию, как обычную переменную (имя без скобок).

In [ ]:
# Пишем вместе:
# 1) функция square(x), возвращающая x^2
# 2) функция apply_twice(g, x), применяющая g дважды: g(g(x))
# 3) напечатать apply_twice(square, 3)


**Списки и досрочный выход из цикла.** `append` добавляет элемент в конец списка, `values[-1]` — последний элемент, `len(values)` — длина, `break` — досрочный выход из цикла.

In [ ]:
# Пишем вместе:
# 1) пустой список values
# 2) цикл по k от 0 до 19: добавлять 2**k в список; выйти из цикла, как только последний элемент > 100
# 3) напечатать список, его последний элемент и длину


## 1. Постановка задачи и локализация корней

Требуется найти $x_*$, такое что $f(x_*) = 0$.

Решение обычно состоит из двух этапов:
1. **Локализация:** найти отрезок $[a, b]$, на котором $f(a)\,f(b) < 0$. Если $f$ непрерывна, на нём есть корень.
2. **Уточнение:** итерационный метод, который приближает корень с нужной точностью.

Весь семинар будем использовать два примера:
- $f(x) = x^2 - 2$, корень $\sqrt{2} \approx 1.41421356$;
- $f(x) = x - \cos x$.

In [ ]:
# Пишем вместе:
# 1) функция f_sq(x) = x^2 - 2
# 2) график на [0, 2] и горизонтальная линия y = 0 (plt.axhline)
# 3) значения функции на концах отрезка [1, 2]


**Мини-задача.** Задайте функцию `f_cos(x)` $= x - \cos x$, постройте её график на $[-2, 2]$ и найдите отрезок длины 1, на концах которого функция имеет разные знаки. Проверьте знаки, напечатав значения на концах.

In [ ]:
# Ваш код


## 2. Метод деления пополам

**Условия:** $f$ непрерывна на $[a, b]$, $f(a)\,f(b) < 0$.

**Шаг:** $c = \dfrac{a + b}{2}$. Если $f(a)\,f(c) \le 0$, корень на $[a, c]$, иначе на $[c, b]$.

**Оценка:** после $n$ шагов длина отрезка равна $\dfrac{b - a}{2^n}$, поэтому для точности $\varepsilon$ достаточно

$$n \ge \log_2\frac{b - a}{\varepsilon}.$$

| Достоинства | Недостатки |
|---|---|
| Всегда сходится, нужна только непрерывность | Медленный: ошибка уменьшается примерно в 2 раза за шаг (линейная сходимость) |
| Простой | Нужен отрезок со сменой знака; не находит корни чётной кратности |

Функция будет возвращать список всех середин отрезков: последний элемент — ответ, длина списка — число итераций. Ограничение `max_iter` защищает от бесконечного цикла.

In [ ]:
# Пишем вместе:
# функция bisection(f, a, b, tol=1e-10, max_iter=100):
#   список mids; цикл не более max_iter раз:
#     середина c, добавить в mids;
#     если f(a) * f(c) <= 0, то b = c, иначе a = c;
#     если b - a < tol, выйти из цикла
#   вернуть mids


In [ ]:
# Пишем вместе:
# корень f_sq на [1, 2] с точностью 1e-6: последнее приближение и число итераций


In [ ]:
# Пишем вместе:
# ошибка каждого приближения |mids - sqrt(2)| и её график в логарифмическом масштабе (plt.semilogy)


**Мини-задача.** Найдите корень `f_cos` на $[0, 1]$ с точностью $10^{-8}$. Сначала оцените число итераций по формуле (`np.log2`), затем сравните с длиной списка.

In [ ]:
# Ваш код


## 3. Метод простой итерации

Уравнение записывается в виде $x = F(x)$ (поиск **неподвижной точки**), итерации:

$$x_{k+1} = F(x_k).$$

**Теорема о сжатии.** Если $F$ отображает $[a, b]$ в себя и $|F'(x)| \le q < 1$ на $[a, b]$, то неподвижная точка единственна и итерации сходятся к ней из любого $x_0 \in [a, b]$.

- Ошибка уменьшается примерно в $|F'(x_*)|$ раз за шаг: чем меньше, тем быстрее.
- Одно уравнение можно записать в виде $x = F(x)$ многими способами, и сходятся не все.
- Критерий остановки: $|x_{k+1} - x_k| < \varepsilon$ (плюс ограничение на число итераций).

In [ ]:
# Пишем вместе:
# функция fixed_point(F, x0, tol=1e-10, max_iter=100):
#   список xs = [x0]; цикл не более max_iter раз:
#     добавить в список F(последний элемент);
#     если |последний - предпоследний| < tol, выйти из цикла
#   вернуть xs


**Пример.** Уравнение $x^2 = 2$ можно переписать двумя способами:

$$F_1(x) = \frac{2}{x}, \qquad F_2(x) = \frac{1}{2}\left(x + \frac{2}{x}\right).$$

У обеих функций неподвижная точка $\sqrt{2}$.

In [ ]:
# Пишем вместе:
# 1) функции F1(x) = 2/x и F2(x) = (x + 2/x)/2
# 2) итерации из x0 = 1 (для F1 не более 10 итераций)
# 3) напечатать оба списка через np.array(...)


**Почему так?** $F_1'(x) = -2/x^2$, в корне $F_1'(\sqrt{2}) = -1$: это не сжатие. $F_2'(x) = \frac{1}{2}\left(1 - \frac{2}{x^2}\right)$, в корне $F_2'(\sqrt{2}) = 0$.

**Наглядно** (просто запустите). Итерации изображаются «лестницей»: от точки на графике $y = F(x)$ горизонтально к прямой $y = x$, затем вертикально снова к графику.

In [ ]:
def cobweb(F, x0, n_steps, x_min, x_max, ax):
    x_line = np.linspace(x_min, x_max, 400)
    ax.plot(x_line, F(x_line), label='y = F(x)')
    ax.plot(x_line, x_line, 'k--', label='y = x')
    x = x0
    for k in range(n_steps):
        y = F(x)
        ax.plot([x, x], [x, y], 'r', lw=1)
        ax.plot([x, y], [y, y], 'r', lw=1)
        x = y
    ax.set_xlim(x_min, x_max)
    ax.set_ylim(x_min, x_max)
    ax.set_aspect('equal')
    ax.legend()


fig, ax = plt.subplots(1, 2, figsize=(11, 5))
cobweb(np.cos, 0.1, 20, 0.0, 1.2, ax[0])
ax[0].set_title('F(x) = cos x: сходится')
cobweb(F1, 1.0, 6, 0.5, 2.5, ax[1])
ax[1].set_title('F(x) = 2/x: зацикливается')
plt.show()

**Мини-задача.** Уравнение $x = \cos x$ можно записать в виде $x = F(x)$, например, так:

$$F_a(x) = \cos x, \qquad F_b(x) = \frac{x + \cos x}{2}, \qquad F_c(x) = 2x - \cos x.$$

1. Для каждой функции запустите `fixed_point` из $x_0 = 1$ с `tol=1e-8` и напечатайте число итераций и последнее значение.
2. Вычислите $|F'(x_*)|$ для каждой из функций при $x_* \approx 0.739$ и объясните результат.

In [ ]:
# Ваш код


## 4. Метод Ньютона

Следующее приближение — точка пересечения касательной в $x_k$ с осью абсцисс:

$$x_{k+1} = x_k - \frac{f(x_k)}{f'(x_k)}.$$

- Это простая итерация с $F(x) = x - \dfrac{f(x)}{f'(x)}$. Для простого корня $F'(x_*) = 0$, поэтому сходимость **квадратичная**:
$$x_{k+1} - x_* \approx \frac{f''(x_*)}{2f'(x_*)}(x_k - x_*)^2,$$
то есть число верных знаков примерно удваивается за шаг.
- Для корня кратности $\alpha$: $F'(x_*) = \dfrac{\alpha - 1}{\alpha}$, сходимость линейная.
- Недостатки: нужна производная; сходимость гарантирована только при $x_0$, достаточно близком к корню.

In [ ]:
# Иллюстрация: два шага метода Ньютона для f(x) = x^2 - 2 из x0 = 2.5 (просто запустите)
x_ill = np.linspace(0.8, 2.7, 400)
plt.plot(x_ill, f_sq(x_ill), 'k', label='f(x)')
plt.axhline(0, color='k', lw=0.8)
x_k = 2.5
for k in range(2):
    x_next = x_k - f_sq(x_k) / (2 * x_k)
    plt.plot([x_k, x_next], [f_sq(x_k), 0], 'r')
    plt.plot([x_k, x_k], [0, f_sq(x_k)], 'r:')
    plt.plot(x_k, f_sq(x_k), 'ro')
    plt.annotate(f'$x_{k}$', (x_k, 0), textcoords='offset points', xytext=(0, -15), ha='center')
    x_k = x_next
plt.annotate('$x_2$', (x_k, 0), textcoords='offset points', xytext=(0, -15), ha='center')
plt.legend()
plt.show()

In [ ]:
# Пишем вместе:
# функция newton(f, df, x0, tol=1e-12, max_iter=50):
#   устроена как fixed_point, но новое приближение x - f(x) / df(x)


In [ ]:
# Пишем вместе:
# 1) производная df_sq(x) = 2x
# 2) метод Ньютона для f_sq из x0 = 1, напечатать приближения


**Вопрос.** Сравните с `xs_F2` из раздела 3. Почему получились те же самые числа? Подставьте $f(x) = x^2 - 2$ в формулу Ньютона.

Посмотрим на ошибки.

In [ ]:
# Пишем вместе:
# ошибки |xs_newton_sq - sqrt(2)|


**Мини-задача.** Найдите корень `f_cos` методом Ньютона из $x_0 = 1$. Для этого задайте производную `df_cos(x)`. Сколько понадобилось итераций?

In [ ]:
# Ваш код


### Когда метод Ньютона не работает (просто запустите)

**Зацикливание.** $f(x) = x^3 - 2x + 2$, $x_0 = 0$:

In [ ]:
xs_cycle = newton(lambda x: x**3 - 2*x + 2, lambda x: 3*x**2 - 2, 0.0, max_iter=8)
print(np.array(xs_cycle))

**Расходимость.** $f(x) = \operatorname{arctg} x$ (корень $x_* = 0$), $x_0 = 1.5$:

In [ ]:
xs_atan = newton(np.arctan, lambda x: 1 / (1 + x**2), 1.5, max_iter=6)
print(np.array(xs_atan))

**Кратный корень.** $f(x) = (x - 1)^2$, $x_0 = 2$. Деление пополам здесь неприменимо: знак функции не меняется. Ньютон сходится, но линейно:

In [ ]:
xs_double = newton(lambda x: (x - 1)**2, lambda x: 2 * (x - 1), 2.0, max_iter=100)
err_double = np.abs(np.array(xs_double) - 1)
print('итераций:', len(xs_double) - 1)
print('первые ошибки:', err_double[:6])
print('отношения соседних ошибок:', err_double[1:6] / err_double[:5])

Отношение ошибок равно $1/2 = (\alpha - 1)/\alpha$ при $\alpha = 2$.

## 5. Метод секущих

Производная заменяется разностным отношением по двум последним приближениям:

$$x_{k+1} = x_k - f(x_k)\,\frac{x_k - x_{k-1}}{f(x_k) - f(x_{k-1})}.$$

- Нужны две начальные точки, производная не нужна.
- Порядок сходимости $\dfrac{1 + \sqrt{5}}{2} \approx 1.618$: ниже, чем у Ньютона, но выше линейного.
- За шаг одно новое вычисление $f$ (у Ньютона $f$ и $f'$), поэтому часто метод секущих выгоднее.

In [ ]:
# Пишем вместе:
# функция secant(f, x0, x1, tol=1e-12, max_iter=50):
#   список xs = [x0, x1]; в цикле берём два последних элемента и добавляем новое приближение по формуле


In [ ]:
# Пишем вместе:
# метод секущих для f_sq из точек 1 и 2, напечатать ошибки


**Мини-задача.** Найдите корень `f_cos` методом секущих из точек $0$ и $1$. Сколько понадобилось итераций (не считая двух начальных точек)?

In [ ]:
# Ваш код


## 6. Сравнение методов. Метод Брента и `scipy`

Ошибки всех методов для уравнения $x = \cos x$ на одном графике (просто запустите; используются ваши функции).

In [ ]:
from scipy.optimize import brentq

root_cos = brentq(f_cos, 0.0, 1.0, xtol=1e-15)

def df_cos_ref(x):
    return 1 + np.sin(x)

results = {
    'деление пополам': bisection(f_cos, 0.0, 1.0, tol=1e-14),
    'простая итерация, F = cos x': fixed_point(np.cos, 1.0, tol=1e-14),
    'Ньютон': newton(f_cos, df_cos_ref, 1.0),
    'секущие': secant(f_cos, 0.0, 1.0),
}
for name, xs in results.items():
    err = np.abs(np.array(xs) - root_cos) + 1e-17
    plt.semilogy(err, 'o-', ms=3, label=name)
plt.ylim(1e-16, 10)
plt.xlim(0, 50)
plt.xlabel('номер итерации')
plt.ylabel('ошибка')
plt.legend()
plt.show()

Линейная сходимость в логарифмическом масштабе — прямая линия. У Ньютона и секущих график круто уходит вниз.

### Метод Брента

Объединяет надёжность деления пополам и скорость интерполяционных методов:
- всегда хранится отрезок со сменой знака;
- на каждом шаге пробуется «быстрый» шаг: секущая или обратная квадратичная интерполяция;
- если быстрый шаг выводит за отрезок или сокращает его слишком медленно, выполняется шаг деления пополам.

Итог: сходимость гарантирована, как у деления пополам, а вблизи корня скорость как у интерполяционных методов.

**Обратная квадратичная интерполяция:** по трём последним точкам строится многочлен второй степени $x = p(y)$ ($x$ как функция от $y$), новое приближение $x_{\text{new}} = p(0)$:

$$x_{\text{new}} = \frac{x_0 f_1 f_2}{(f_0 - f_1)(f_0 - f_2)} + \frac{x_1 f_0 f_2}{(f_1 - f_0)(f_1 - f_2)} + \frac{x_2 f_0 f_1}{(f_2 - f_0)(f_2 - f_1)}.$$

В `scipy` метод Брента реализован функцией `scipy.optimize.brentq(f, a, b)`. На практике для одного уравнения с известным отрезком смены знака обычно используют её. Метод Ньютона и метод секущих есть в `scipy.optimize.newton`.

In [ ]:
# Пишем вместе:
# brentq для f_cos на [0, 1] с full_output=True: корень, число итераций и вычислений функции


**Мини-задача.** Найдите с помощью `brentq` корень $\operatorname{arctg} x$ на отрезке $[-1, 1.5]$ — функции, на которой метод Ньютона из $x_0 = 1.5$ расходился.

In [ ]:
# Ваш код


---
## 7. Задачи

### Основные

**Задача 1.** Найдите корень уравнения $e^x = 2$ методом деления пополам на $[0, 1]$ и сравните с `np.log(2)`.

In [ ]:
# Ваш код


**Задача 2.** Уравнение $x^3 - x - 1 = 0$: постройте график, выберите отрезок со сменой знака, найдите корень делением пополам и проверьте ответ с помощью `brentq`.

In [ ]:
# Ваш код


**Задача 3.** Уравнение $\sin x = 0.5x$ имеет три корня. Постройте график $f(x) = \sin x - 0.5x$ на $[-3, 3]$, выделите три отрезка, каждый из которых содержит ровно один корень, и найдите все корни делением пополам.

In [ ]:
# Ваш код


**Задача 4.** Вычислите $\sqrt[3]{10}$ методом Ньютона. Какое уравнение нужно решать? Сравните с `10 ** (1/3)`.

In [ ]:
# Ваш код


**Задача 5.** Для уравнения $x^3 - x - 1 = 0$ сравните число итераций деления пополам на $[1, 2]$ и метода Ньютона из $x_0 = 1.5$ при одинаковой точности `tol=1e-10`.

In [ ]:
# Ваш код


**Задача 6. Возвращаемся к разминке.** Найдите $x$, при котором сплайн `cs_table` из разминки равен $0.5$: решите уравнение $S(x) - 0.5 = 0$ делением пополам и с помощью `brentq` (отрезок выберите по таблице или по графику). Сравните ответ с $\pi/6$. Почему ответы немного отличаются?

In [ ]:
# Ваш код


### Для тех, кто справился быстрее

**Задача 7. Уравнение Кеплера.** Положение тела на эллиптической орбите в момент времени задаётся эксцентрической аномалией $E$, которая находится из уравнения

$$E - e\sin E = M,$$

где $e$ — эксцентриситет орбиты, $M$ — средняя аномалия (пропорциональна времени). Найдите $E$ методом Ньютона при $e = 0.5$, $M = 1$ (начальное приближение $E_0 = M$).

In [ ]:
# Ваш код


**Задача 8. Задача Стефана.** При плавлении вещества фронт плавления движется по закону $s(t) = 2\lambda\sqrt{a t}$, где $a$ — температуропроводность, а безразмерный параметр $\lambda$ находится из уравнения

$$\lambda\, e^{\lambda^2} \operatorname{erf}\lambda = \frac{\mathrm{St}}{\sqrt{\pi}}.$$

Здесь $\mathrm{St}$ — число Стефана (отношение теплоты нагрева к теплоте плавления), $\operatorname{erf}$ — функция ошибок (`from scipy.special import erf`).

1. Найдите $\lambda$ для $\mathrm{St} = 0.1,\ 0.5,\ 1,\ 2$ любым методом.
2. При малых $\lambda$ имеем $e^{\lambda^2} \approx 1$, $\operatorname{erf}\lambda \approx 2\lambda/\sqrt{\pi}$, откуда $\lambda \approx \sqrt{\mathrm{St}/2}$. Сравните с найденными значениями. При каких St приближение хорошее?

In [ ]:
from scipy.special import erf

# Ваш код


---
## Итоги

| Метод | Что нужно | Сходимость | Надёжность |
|---|---|---|---|
| Деление пополам | отрезок со сменой знака | линейная, в 2 раза за шаг | всегда сходится |
| Простая итерация | форма $x = F(x)$ с $\lvert F'\rvert < 1$ | линейная, в $\lvert F'(x_*)\rvert$ раз за шаг | зависит от выбора $F$ |
| Ньютон | $f'$, хорошее начальное приближение | квадратичная (простой корень) | локальная |
| Секущие | две начальные точки | порядок $\approx 1.618$ | локальная |
| Брент | отрезок со сменой знака | быстрая вблизи корня | всегда сходится |

---
## 8. Домашнее задание

### Основная часть

**ДЗ 1. Кратный корень.** Для $f(x) = (x - 1)^3(x + 2)$:
1. Найдите корень $x_* = 1$ методом Ньютона из $x_0 = 2$. Сколько понадобилось итераций? Чему равно отношение соседних ошибок и как оно связано с кратностью корня?
2. Модифицированный метод Ньютона для корня кратности $\alpha$: $x_{k+1} = x_k - \alpha\dfrac{f(x_k)}{f'(x_k)}$. Реализуйте его и сравните число итераций.
3. Можно ли найти этот корень делением пополам? Почему?

**ДЗ 2. Уравнение Кеплера.** Для $e = 0.5$ найдите $E(M)$ для 50 значений $M$ на $[0, 2\pi]$ и постройте график. Затем повторите для $e = 0.99$ (начальное приближение $E_0 = M$, `max_iter=100`) и напечатайте число итераций для каждого $M$. При каких $M$ метод работает плохо и почему? Подсказка: посмотрите на $f'(E) = 1 - e\cos E$ вблизи $E = 0$. Проверьте, помогает ли начальное приближение $E_0 = \pi$. Как можно использовать `brentq` (какой отрезок взять)?

**ДЗ 3. Максимум плотности воды** (продолжение ДЗ первого семинара). Постройте по таблице плотности воды кубический сплайн и найдите максимум плотности как корень $S'(T) = 0$ своими методами: делением пополам и методом Ньютона (производные сплайна: `cs(T, 1)` и `cs(T, 2)`). Сравните с результатом `brentq`.

### Задачи повышенной сложности (по желанию)

**ДЗ 4. Фрактал Ньютона.** Уравнение $z^3 = 1$ имеет три комплексных корня. Возьмите сетку начальных точек $z_0 = x + iy$, $x, y \in [-2, 2]$ (например, 500 × 500), выполните 30 итераций метода Ньютона сразу для всех точек (с массивами `numpy` это делается без циклов по точкам) и раскрасьте каждую начальную точку в цвет корня, к которому она сошлась (`plt.imshow`). Объясните, что видно на картинке.

**ДЗ 5. Поиск всех корней.** Многочлен $P(x) = x^4 - 10x^3 + 35x^2 - 50x + 24$ имеет четыре корня. Найдите их методом удаления корней (см. дополнительный раздел): после нахождения корня $x_1$ ищите корни функции $P(x)/(x - x_1)$ и т. д. Сравните с `np.roots`.

---
## Дополнительно

**Метод парабол.** Через три последние точки $x_{k-2}, x_{k-1}, x_k$ проводится интерполяционная парабола (в форме Ньютона)

$$p_2(x) = f_k + f[x_{k-1}, x_k](x - x_k) + f[x_{k-2}, x_{k-1}, x_k](x - x_k)(x - x_{k-1}),$$

и в качестве $x_{k+1}$ берётся её корень, ближайший к $x_k$. Порядок сходимости выше, чем у секущих, но ниже, чем у Ньютона. Даже при вещественных $f$ и начальных точках метод может привести к комплексному корню, поэтому удобен для поиска корней многочленов.

**Поиск всех корней (удаление корней).** Если найден корень $x_1$ функции $f$, то остальные корни $f$ — это корни функции

$$f_1(x) = \frac{f(x)}{x - x_1}.$$

Найдя корень $x_2$ функции $f_1$, переходим к $f_2(x) = \dfrac{f(x)}{(x - x_1)(x - x_2)}$ и т. д. Поскольку корни известны лишь приближённо, ошибки накапливаются; поэтому найденное по вспомогательной функции приближение уточняют несколькими итерациями по исходной функции $f$.